# Импорты и пути

In [ ]:
from pathlib import Path
import sys
import random
from collections import Counter
from itertools import product

import pandas as pd
from openai import AsyncOpenAI
from transformers import AutoTokenizer

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from generation import GenerationResult, generate_one
from io_utils import (
    load_json,
    load_jsonl,
    load_checkpoint,
    save_checkpoint,
    load_results,
    save_json,
)
from prompts import build_messages
from validation import ValidationReason


DATA_PATH = PROJECT_ROOT / "data" / "dev.jsonl"

EXPERIMENT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "experiments"
    / "parameters"
)

CACHE_DIR = EXPERIMENT_DIR / "cache"
DIAGNOSTICS_DIR = EXPERIMENT_DIR / "diagnostics"
RESULTS_PATH = EXPERIMENT_DIR / "results.txt"
SAMPLE_PATH = EXPERIMENT_DIR / "experiment_sample.json"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

# Создание клиента

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

client = AsyncOpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
)

In [ ]:
response = await client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "user", "content": "Ответь кратко: какое слово считается самым смешным?"}
    ],
    temperature=0.7,
)

print(response.choices[0].message.content)

# Тестовая выборка из 20 товаров

- выборка из `dev.jsonl`;
- в тест входят только те товары, которые могут быть обработаны без ошибки long_data (вход <1500 токенов).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def count_input_tokens(product: dict) -> int:
    """Считает токены полного входа модели."""
    messages = build_messages(source_product=product)

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True
    )

    return len(input_ids)


In [ ]:
dev_data = load_jsonl(DATA_PATH)

if SAMPLE_PATH.exists():
    experiment_products = load_json(SAMPLE_PATH)

else:
    short_products = [
        product
        for product in dev_data
        if count_input_tokens(product) <= 1500
    ]

    rng = random.Random(42)

    experiment_products = rng.sample(
        short_products,
        20,
    )

    save_json(
        SAMPLE_PATH,
        experiment_products,
    )

print("Товаров в dev:", len(dev_data))
print("Товаров в эксперименте:", len(experiment_products))

# Обработка всех товаров

In [ ]:
async def run_products(
    products: list[dict],
    generation_params: dict,
    experiment_name: str,
) -> list[GenerationResult]:
    """Обрабатывает набор товаров с checkpoint для одной конфигурации."""

    cache_path = CACHE_DIR / f"{experiment_name}.jsonl"
    diagnostic_dir = DIAGNOSTICS_DIR / experiment_name

    processed_ids = load_checkpoint(cache_path)

    for product in products:
        if product["product_id"] in processed_ids:
            continue

        result = await generate_one(
            client=client,
            model_name=model_name,
            source_product=product,
            generation_params=generation_params,
            max_attempts=2,
            diagnostic_dir=diagnostic_dir,
        )

        save_checkpoint(
            cache_path,
            result.model_dump(mode="json"),
        )

    return [
        GenerationResult.model_validate(item)
        for item in load_results(cache_path)
    ]

# Расчет метрик

In [ ]:
def summarize_experiment(
    experiment_name: str,
    generation_params: dict,
    results: list[GenerationResult],
) -> dict:
    """Считает метрики одной конфигурации генерации."""

    valid_results = [
        result
        for result in results
        if result.valid and result.card is not None
    ]

    n = len(results)

    input_tokens = sum(
        result.input_tokens
        for result in results
    )

    output_tokens = sum(
        result.output_tokens
        for result in results
    )

    avg_description_length = (
        sum(
            len(result.card.description)
            for result in valid_results
        )
        / len(valid_results)
        if valid_results
        else 0
    )

    output_truncated = sum(
        ValidationReason.OUTPUT_TRUNCATED in result.errors
        for result in results
    )

    return {
        "experiment": experiment_name,
        "top_k": generation_params.get("top_k", "без ограничения"),
        "repetition_penalty": generation_params.get(
            "repetition_penalty",
            1.0,
        ),
        "max_tokens": generation_params["max_tokens"],
        "valid_rate": (
            len(valid_results) / n
            if n
            else 0
        ),
        "avg_attempts": (
            sum(result.attempts for result in results) / n
            if n
            else 0
        ),
        "avg_description_length": avg_description_length,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "avg_input_tokens": input_tokens / n if n else 0,
        "avg_output_tokens": output_tokens / n if n else 0,
        "output_truncated": output_truncated,
    }

# Запуск одной конфигурации параметров

In [7]:
async def run_experiment(
    experiment_name: str,
    generation_params: dict,
) -> dict:
    """Запускает одну конфигурацию и возвращает её метрики."""

    results = await run_products(
        products=experiment_products,
        generation_params=generation_params,
        experiment_name=experiment_name,
    )

    return summarize_experiment(
        experiment_name=experiment_name,
        generation_params=generation_params,
        results=results,
    )

# Сетка параметров

In [ ]:
BASE_PARAMS = {
    "temperature": 0.4,
    "top_p": 0.9,
    "max_tokens": 500,
    "seed": 42,
}

top_k_values = [
    None,
    20,
    50,
]

repetition_penalty_values = [
    1.0,
    1.1,
    1.2,
]

parameter_results = []

for top_k, repetition_penalty in product(
    top_k_values,
    repetition_penalty_values,
):
    params = BASE_PARAMS.copy()

    if top_k is not None:
        params["top_k"] = top_k

    params["repetition_penalty"] = repetition_penalty

    top_k_name = (
        "unlimited"
        if top_k is None
        else str(top_k)
    )

    repetition_name = str(
        repetition_penalty
    ).replace(".", "")

    experiment_name = (
        f"top_k_{top_k_name}"
        f"_repetition_{repetition_name}"
    )

    summary = await run_experiment(
        experiment_name=experiment_name,
        generation_params=params,
    )

    parameter_results.append(summary)


parameter_df = pd.DataFrame(parameter_results)

display(
    parameter_df[
        [
            "top_k",
            "repetition_penalty",
            "valid_rate",
            "avg_description_length",
            "avg_input_tokens",
            "avg_output_tokens",
            "avg_attempts",
            "output_truncated",
        ]
    ]
)

parameter_df.to_csv(
    RESULTS_PATH,
    sep="\t",
    index=False,
)

In [ ]:
SELECTED_TOP_K = None
SELECTED_REPETITION_PENALTY = 1.1

# Выбор max_tokens

In [ ]:
max_tokens_values = [
    250,
    350,
    500,
]

max_tokens_results = []

for max_tokens in max_tokens_values:
    params = {
        "temperature": 0.4,
        "top_p": 0.9,
        "repetition_penalty": SELECTED_REPETITION_PENALTY,
        "max_tokens": max_tokens,
        "seed": 42,
    }

    if SELECTED_TOP_K is not None:
        params["top_k"] = SELECTED_TOP_K

    top_k_name = (
        "unlimited"
        if SELECTED_TOP_K is None
        else str(SELECTED_TOP_K)
    )

    repetition_name = str(
        SELECTED_REPETITION_PENALTY
    ).replace(".", "")

    experiment_name = (
        f"max_tokens_{max_tokens}"
        f"_top_k_{top_k_name}"
        f"_repetition_{repetition_name}"
    )

    summary = await run_experiment(
        experiment_name=experiment_name,
        generation_params=params,
    )

    max_tokens_results.append(summary)


max_tokens_df = pd.DataFrame(max_tokens_results)

display(
    max_tokens_df[
        [
            "max_tokens",
            "valid_rate",
            "avg_description_length",
            "avg_input_tokens",
            "avg_output_tokens",
            "avg_attempts",
            "output_truncated",
        ]
    ]
)

# Запись результатов

In [ ]:
all_results_df = pd.concat(
    [
        parameter_df,
        max_tokens_df,
    ],
    ignore_index=True,
)

all_results_df.to_csv(
    RESULTS_PATH,
    sep="\t",
    index=False,
)

all_results_df